# Model Training: Logistic Regression And XGBoost

This notebook trains two probability-of-default models on the same train-validation split: a regularized logistic regression baseline and an XGBoost benchmark. The goal is to produce comparable validation predictions that can be evaluated in the next notebook.

## Modeling Notes

Logistic regression is a natural credit-risk baseline because it maps a linear borrower score into a probability of default. It is interpretable through its coefficients, but it is sensitive to feature scale and extreme values.

- **Scaling** puts numeric variables on comparable units before fitting the model.
- **L2 regularization** shrinks large coefficients and improves stability, especially when variables are noisy or correlated.
- **Quantile clipping** is fitted on the training set only and limits extreme feature values before scaling. It is optional and can be disabled later with `clip_quantiles=None`.

XGBoost is used as a nonlinear benchmark for tabular data. It does not need feature scaling, but it is trained on the same cleaned feature matrix so the comparison uses the same validation sample. The benchmark is left unweighted by default so its predicted probabilities remain closer to raw PD estimates; class weighting can be tested later as a ranking-focused experiment.

## Setup

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

TABLES_DIR = PROJECT_ROOT / "reports" / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

from load_data import (
    fit_missing_value_preprocessor,
    load_training_data,
    make_train_val_split,
    prepare_features_target,
)
from logistic_model import LogisticPDModel
from xgboost_model import XGBoostPDModel

pd.set_option("display.float_format", "{:.4f}".format)

## Load Data And Build Split

The validation split is stratified so the default rate is preserved. Missing-value imputation is fitted on the training features only, then applied to both training and validation data.

In [ ]:
df = load_training_data()
X, y = prepare_features_target(df)
X_train, X_val, y_train, y_val = make_train_val_split(X, y)

missing_preprocessor = fit_missing_value_preprocessor(X_train)
X_train_clean = missing_preprocessor.transform(X_train)
X_val_clean = missing_preprocessor.transform(X_val)

{
    "train_shape": X_train_clean.shape,
    "validation_shape": X_val_clean.shape,
    "train_default_rate": y_train.mean(),
    "validation_default_rate": y_val.mean(),
    "missing_indicator_columns": missing_preprocessor.indicator_columns,
}

## Train Models

The logistic regression model uses L2 regularization, training-fitted clipping, and standardization. The XGBoost model uses histogram-based tree building on CUDA; set `XGBOOST_DEVICE = "cpu"` if running on a machine without a compatible GPU.

In [ ]:
XGBOOST_DEVICE = "cuda"

models = {
    "logistic_l2_clipped": LogisticPDModel(
        penalty="l2",
        C=1.0,
        clip_quantiles=(0.01, 0.99),
        class_weight=None,
    ),
    "xgboost": XGBoostPDModel(device=XGBOOST_DEVICE),
}

metrics = []
validation_predictions = pd.DataFrame(
    {
        "row_id": X_val.index,
        "y_true": y_val.to_numpy(),
    }
)

for model_name, model in models.items():
    model.fit(X_train_clean, y_train)
    validation_predictions[f"{model_name}_pd"] = model.predict_proba(X_val_clean)

    model_metrics = model.evaluate(X_val_clean, y_val)
    model_metrics["model"] = model_name
    metrics.append(model_metrics)

metrics_table = (
    pd.DataFrame(metrics)
    .loc[:, ["model", "roc_auc", "pr_auc", "log_loss"]]
    .sort_values("roc_auc", ascending=False)
    .reset_index(drop=True)
)

metrics_table

## Logistic Regression Coefficients

Because the logistic model is fitted after standardization, the odds ratios below show the multiplicative change in odds for a one-standard-deviation increase in each feature, after clipping and missing-value preprocessing.

In [ ]:
logistic_coefficients = models["logistic_l2_clipped"].coefficients()

logistic_coefficients.head(12)

## Save Training Outputs

The evaluation notebook will use these validation predictions and metrics to build ROC curves, precision-recall curves, calibration plots, PD distributions, and decile analysis.

In [ ]:
metrics_path = TABLES_DIR / "model_metrics.csv"
predictions_path = TABLES_DIR / "validation_predictions.csv"
coefficients_path = TABLES_DIR / "logistic_coefficients.csv"

metrics_table.to_csv(metrics_path, index=False)
validation_predictions.to_csv(predictions_path, index=False)
logistic_coefficients.to_csv(coefficients_path, index=False)

{
    "metrics_path": str(metrics_path.relative_to(PROJECT_ROOT)),
    "predictions_path": str(predictions_path.relative_to(PROJECT_ROOT)),
    "coefficients_path": str(coefficients_path.relative_to(PROJECT_ROOT)),
}